# 01 — CNN Fake Review Detection

Implements the CNN from Jayasinghe & Dassanayaka (2025), *Detecting deception: employing deep neural networks for fraudulent review detection on Amazon*:

- Text cleaning → NLTK **TweetTokenizer** → integer sequences → padding
- **GloVe 6B 100d** pre-trained embeddings
- Conv1D(128, k=5) → MaxPool(2) → Conv1D(64, k=5) → MaxPool(2) → Conv1D(32, k=5) → GlobalMaxPool
- Concatenated with engineered features (rating, review length, review/summary sentiment, rating–sentiment difference)
- Dense(64, ReLU) → Dropout(0.5) → Sigmoid
- **HACE loss** (Hybrid Adaptive Cross-Entropy): dynamic class weight, label smoothing α = 0.1, focusing γ = 2
- 10 epochs

Runtime: **GPU (T4)**. Outputs go to `outputs/` in Drive.

In [ ]:
# --- Setup: mount Google Drive (dataset lives there, not in GitHub) ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FakeReviewProject'   # change if your folder differs
DATA_DIR    = f'{PROJECT_DIR}/dataset'                     # train.csv, test.csv
OUT_DIR     = f'{PROJECT_DIR}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))

In [ ]:
!pip -q install nltk
import tensorflow as tf, keras
print(tf.__version__, keras.__version__, tf.config.list_physical_devices('GPU'))
SEED = 42
keras.utils.set_random_seed(SEED)

In [ ]:
import pandas as pd, numpy as np, re

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

def prepare(df):
    # Module 1/2 cleaning: drop rows without text/label, fill missing summary
    df = df.dropna(subset=['review_text', 'label']).copy()
    df['review_summary'] = df['review_summary'].fillna('').astype(str)
    df['review_text'] = df['review_text'].astype(str)
    df['label'] = df['label'].astype(int)
    return df.reset_index(drop=True)

train_df, test_df = prepare(train_df), prepare(test_df)
print(train_df.shape, test_df.shape)
print(train_df.label.value_counts(), test_df.label.value_counts())

## Text preprocessing (paper §3.1.3): cleaning, normalisation, TweetTokenizer

In [ ]:
from nltk.tokenize import TweetTokenizer
from collections import Counter

tok = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
URL, HTML, NONTXT = re.compile(r'https?://\S+|www\.\S+'), re.compile(r'<[^>]+>'), re.compile(r"[^a-z0-9'!?.,:;()\s]")

def clean(text):
    text = str(text).lower()
    text = URL.sub(' ', text); text = HTML.sub(' ', text); text = NONTXT.sub(' ', text)
    return text

def to_tokens(df):
    return [tok.tokenize(clean(s + ' . ' + t)) for s, t in zip(df.review_summary, df.review_text)]

train_tok, test_tok = to_tokens(train_df), to_tokens(test_df)

MAX_WORDS, MAX_LEN = 50000, 200
counts = Counter(w for toks in train_tok for w in toks)
vocab = {w: i + 2 for i, (w, _) in enumerate(counts.most_common(MAX_WORDS - 2))}   # 0=pad, 1=OOV

def encode(tok_lists):
    X = np.zeros((len(tok_lists), MAX_LEN), dtype='int32')
    for i, toks in enumerate(tok_lists):
        ids = [vocab.get(w, 1) for w in toks][:MAX_LEN]
        X[i, :len(ids)] = ids
    return X

X_train_txt, X_test_txt = encode(train_tok), encode(test_tok)
print(X_train_txt.shape, 'median tokens:', np.median([len(t) for t in train_tok]))

## Engineered features (paper Table 1, selected with SelectKBest)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

FEATURES = ['review_length', 'review_sentiment', 'rating_sentiment_difference',
            'summary_sentiment', 'rating', 'summary_length']
skb = SelectKBest(f_classif, k='all').fit(train_df[FEATURES], train_df.label)
print(pd.DataFrame({'feature': FEATURES, 'F-score': skb.scores_, 'p-value': skb.pvalues_})
        .sort_values('F-score', ascending=False))

NUM_FEATURES = ['rating', 'review_length', 'review_sentiment', 'summary_sentiment', 'rating_sentiment_difference']
scaler = StandardScaler().fit(train_df[NUM_FEATURES])
X_train_num = scaler.transform(train_df[NUM_FEATURES]).astype('float32')
X_test_num  = scaler.transform(test_df[NUM_FEATURES]).astype('float32')
y_train, y_test = train_df.label.values, test_df.label.values

In [ ]:
from sklearn.model_selection import train_test_split
idx_tr, idx_val = train_test_split(np.arange(len(y_train)), test_size=0.1, stratify=y_train, random_state=SEED)
tr = ([X_train_txt[idx_tr], X_train_num[idx_tr]], y_train[idx_tr])
va = ([X_train_txt[idx_val], X_train_num[idx_val]], y_train[idx_val])

## GloVe 6B 100d embeddings

In [ ]:
GLOVE = '/content/glove.6B.100d.txt'
if not os.path.exists(GLOVE):
    !wget -q https://nlp.stanford.edu/data/glove.6B.zip -O /content/glove.6B.zip
    !unzip -q -o /content/glove.6B.zip glove.6B.100d.txt -d /content/

EMB_DIM = 100
emb = np.random.normal(0, 0.05, (MAX_WORDS, EMB_DIM)).astype('float32'); emb[0] = 0
hits = 0
with open(GLOVE, encoding='utf8') as f:
    for line in f:
        w, *vec = line.rstrip().split(' ')
        i = vocab.get(w)
        if i is not None and i < MAX_WORDS:
            emb[i] = np.asarray(vec, dtype='float32'); hits += 1
print(f'GloVe coverage: {hits}/{len(vocab)}')

## HACE loss (paper Eq. 4)

$E = -\frac1N\sum\big[w\,(y(1-\alpha)+\frac{\alpha}{2})(1-p)^\gamma\log p + (1-w)\,((1-y)(1-\alpha)+\frac{\alpha}{2})\,p^\gamma\log(1-p)\big]$

The dynamic weight is recomputed at the end of each epoch from the ratio of correctly identified negatives to positives, $r = N_{neg}/N_{pos}$ (paper §3.2.1). So that $w$ and $1-w$ both stay positive, we use the normalised form $w = r/(1+r)$.

In [ ]:
ALPHA, GAMMA = 0.1, 2.0
pos_rate = y_train[idx_tr].mean()
hace_w = tf.Variable((1 - pos_rate), trainable=False, dtype='float32')   # initial: inverse prior

def hace_loss(y_true, y_pred):
    y_true = tf.cast(y_true, 'float32')
    p = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    pos = hace_w * (y_true * (1 - ALPHA) + ALPHA / 2) * tf.pow(1 - p, GAMMA) * tf.math.log(p)
    neg = (1 - hace_w) * ((1 - y_true) * (1 - ALPHA) + ALPHA / 2) * tf.pow(p, GAMMA) * tf.math.log(1 - p)
    return -tf.reduce_mean(pos + neg)

class HACEWeightUpdate(keras.callbacks.Callback):
    """End of each epoch: w_n = N_negative / N_positive correctly identified (normalised)."""
    def __init__(self, X, y): super().__init__(); self.X, self.y = X, y
    def on_epoch_end(self, epoch, logs=None):
        pred = (self.model.predict(self.X, batch_size=2048, verbose=0).ravel() >= 0.5)
        tn = np.sum((pred == 0) & (self.y == 0)); tp = np.sum((pred == 1) & (self.y == 1))
        r = tn / max(tp, 1)
        hace_w.assign(r / (1 + r))
        print(f' — HACE: TN={tn} TP={tp} -> w={hace_w.numpy():.4f}')

## Model

In [ ]:
from keras import layers

txt_in = keras.Input(shape=(MAX_LEN,), name='text')
num_in = keras.Input(shape=(len(NUM_FEATURES),), name='features')
x = layers.Embedding(MAX_WORDS, EMB_DIM, embeddings_initializer=keras.initializers.Constant(emb), name='glove')(txt_in)
x = layers.Conv1D(128, 5, activation='relu')(x); x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(64, 5, activation='relu')(x);  x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(32, 5, activation='relu')(x);  x = layers.GlobalMaxPooling1D()(x)
x = layers.Concatenate()([x, num_in])
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(1, activation='sigmoid')(x)

cnn = keras.Model([txt_in, num_in], out, name='cnn_fake_review')
cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss=hace_loss,
            metrics=['accuracy', keras.metrics.Precision(name='precision'),
                     keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')])
cnn.summary()

In [ ]:
import time
t0 = time.time()
history = cnn.fit(tr[0], tr[1], validation_data=va, epochs=10, batch_size=256,
                  callbacks=[HACEWeightUpdate(tr[0], tr[1]),
                             keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True)])
train_time = time.time() - t0
print(f'Training time: {train_time:.1f}s')
cnn.save(f'{OUT_DIR}/cnn_model.keras')

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history.history)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h[['loss', 'val_loss']].plot(ax=ax[0], title='HACE loss'); h[['auc', 'val_auc']].plot(ax=ax[1], title='ROC-AUC')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/cnn_training_curves.png', dpi=150); plt.show()

## Evaluation on the test set

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

def evaluate(name, y_true, y_prob, categories, threshold=0.5, extra=None):
    y_pred = (y_prob >= threshold).astype(int)
    m = {
        'model': name,
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),   # fake class
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1_fake':   f1_score(y_true, y_pred, zero_division=0),
        'f1_macro':  f1_score(y_true, y_pred, average='macro'),
        'roc_auc':   roc_auc_score(y_true, y_prob),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'n_test': int(len(y_true)),
    }
    if extra: m.update(extra)
    # per-category results (paper Table 6)
    per_cat = {}
    for c in sorted(pd.unique(categories)):
        idx = np.asarray(categories) == c
        yt, yp, pr = y_true[idx], y_pred[idx], y_prob[idx]
        per_cat[c] = {
            'roc_auc':   roc_auc_score(yt, pr) if len(np.unique(yt)) > 1 else None,
            'precision': precision_score(yt, yp, zero_division=0),
            'recall':    recall_score(yt, yp, zero_division=0),
            'f1_fake':   f1_score(yt, yp, zero_division=0),
        }
    m['per_category'] = per_cat
    print(classification_report(y_true, y_pred, target_names=['Genuine', 'Fake'], digits=4))
    print(pd.DataFrame(per_cat).T.round(4))
    key = name.lower()
    with open(f'{OUT_DIR}/{key}_metrics.json', 'w') as f:
        json.dump(m, f, indent=2)
    np.save(f'{OUT_DIR}/{key}_test_probs.npy', y_prob)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items() if k != 'per_category'})
    return m

In [ ]:
val_prob = cnn.predict(va[0], batch_size=2048, verbose=0).ravel()
# Pick the decision threshold that maximises F1 for the fake class on the validation set (never on test)
ths = np.arange(0.05, 0.96, 0.01)
best_t = ths[np.argmax([f1_score(va[1], val_prob >= t) for t in ths])]
print('Validation-selected threshold:', round(best_t, 2))

t0 = time.time()
test_prob = cnn.predict([X_test_txt, X_test_num], batch_size=2048, verbose=0).ravel()
cnn_metrics = evaluate('CNN', y_test, test_prob, test_df.category.values, threshold=best_t,
                       extra={'threshold': float(best_t), 'train_time_s': train_time,
                              'infer_time_s': time.time() - t0, 'train_size': int(len(idx_tr))})

## Save everything needed for prediction (used by `05_predict.ipynb`)

In [ ]:
import pickle
with open(f'{OUT_DIR}/cnn_vocab.json', 'w') as f:
    json.dump(vocab, f)
with open(f'{OUT_DIR}/cnn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{OUT_DIR}/cnn_config.json', 'w') as f:
    json.dump({'max_len': MAX_LEN, 'max_words': MAX_WORDS, 'num_features': NUM_FEATURES,
               'threshold': float(best_t), 'alpha': ALPHA, 'gamma': GAMMA}, f, indent=2)
print(sorted(os.listdir(OUT_DIR)))

## Predict a new review

In [ ]:
!pip -q install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

def predict_review(text, summary='', rating=5.0):
    rs, ss = vader.polarity_scores(text)['compound'], vader.polarity_scores(summary)['compound']
    feats = pd.DataFrame([{'rating': rating, 'review_length': len(text.split()), 'review_sentiment': rs,
                           'summary_sentiment': ss, 'rating_sentiment_difference': rating / 5 - rs}])
    xt = encode([tok.tokenize(clean(summary + ' . ' + text))])
    p = float(cnn.predict([xt, scaler.transform(feats[NUM_FEATURES]).astype('float32')], verbose=0)[0, 0])
    return ('FAKE' if p >= best_t else 'GENUINE'), round(p, 4)

predict_review('Absolutely amazing product!!! Best purchase ever, everyone should buy this, five stars!!!', 'Amazing', 5)